In [1]:

# --- Imports ---
import os, json, math, glob, pathlib, random
import numpy as np
import torch
import matplotlib.pyplot as plt

# Matplotlib: no custom styles/colors; each plot is its own figure.
print(torch.__version__)


2.8.0+cu128


In [2]:

# --- USER CONFIG ---
# 1) Point this to the JSON produced by rs_cavity_explicit_aw_ard.py
JSON_PATH = "/home/goring/mean_field_langevin/MCMC_sparse/results/plot1/rs_cavity_explicit_aw_ard_P1000_kap7.500e-03_Ptr1000_Peval50000_kap7.500e-03_N512_B1024_g0.5.json"   # <-- CHANGE THIS

# 2) Where to save the plots
OUTDIR = None  # None => "<json_dir>/plots_pwa"; or set a custom directory path

# 3) Joint plots selection:
J_INDICES = None        # e.g., [0,1,2]; or None to auto-pick (variance-based)
TOPK_BY_VAR = 3         # used only if J_INDICES is None

# 4) Downsampling / limits (to keep big runs feasible)
MAX_CKPTS = None        # e.g., 20 to load only the last 20 checkpoints; None => load all
SUBSAMPLE_PER_CKPT = None  # e.g., 20000 to randomly subsample per checkpoint; None => use all
MAX_POINTS_2D = 1_000_000  # cap points in each hexbin (downsampled if necessary)

# 5) Histogram bins
BINS = 300


In [3]:

# --- Helper functions ---

def _resolve_path(base_dir, p):
    """Resolve checkpoint path (supports absolute or relative to JSON)."""
    pp = pathlib.Path(p)
    if pp.is_absolute():
        return str(pp)
    return str((pathlib.Path(base_dir) / pp).resolve())

def load_json_and_ckpts(json_path, max_ckpts=None, subsample_per_ckpt=None, seed=123):
    json_path = str(pathlib.Path(json_path).expanduser())
    with open(json_path, "r") as f:
        meta = json.load(f)

    base_dir = str(pathlib.Path(json_path).parent.resolve())

    ckpt_paths = []
    # periodic snapshots
    for item in meta.get("checkpoints", []):
        p = item.get("path", None)
        if p is None: 
            continue
        ckpt_paths.append(_resolve_path(base_dir, p))
    # final checkpoint
    final = meta.get("final_checkpoint", {})
    if "path" in final:
        ckpt_paths.append(_resolve_path(base_dir, final["path"]))

    # deduplicate while preserving order (favor later entries)
    seen = set()
    uniq = []
    for p in ckpt_paths:
        if p not in seen:
            seen.add(p)
            uniq.append(p)

    # optionally keep only the last max_ckpts
    if max_ckpts is not None and len(uniq) > max_ckpts:
        uniq = uniq[-max_ckpts:]

    # load and aggregate
    rng = np.random.RandomState(seed)
    W_list, a_list = [], []
    for i, p in enumerate(uniq, 1):
        ck = torch.load(p, map_location="cpu")
        W = ck["W"]        # (B,d)
        a = ck["a"][:,0]   # (B,)
        if subsample_per_ckpt is not None and W.shape[0] > subsample_per_ckpt:
            idx = rng.choice(W.shape[0], size=subsample_per_ckpt, replace=False)
            W = W[idx]
            a = a[idx]
        W_list.append(W)
        a_list.append(a)
        print(f"loaded {i:>4d}/{len(uniq)}: {p}  ->  W:{tuple(W.shape)}  a:{tuple(a.shape)}")

    W = torch.cat(W_list, dim=0) if len(W_list) > 1 else W_list[0]
    a = torch.cat(a_list, dim=0) if len(a_list) > 1 else a_list[0]

    return meta, uniq, W, a

def choose_j_indices(W, j_indices=None, topk_by_var=3):
    B, d = W.shape
    if j_indices is not None and len(j_indices) > 0:
        # sanitize into valid range
        jj = sorted(set(int(max(0, min(d-1, int(j)))) for j in j_indices))
        return jj
    # auto-pick: largest empirical variance coordinates
    var = W.float().var(dim=0, unbiased=False)  # (d,)
    topk = min(topk_by_var, d)
    vals, idx = torch.topk(var, k=topk)
    auto = sorted(idx.tolist())
    print(f"Auto-picked j by variance (top {topk}): {auto}")
    return auto

def ensure_outdir(json_path, outdir=None):
    if outdir is not None:
        os.makedirs(outdir, exist_ok=True)
        return outdir
    base = pathlib.Path(json_path).parent / "plots_pwa"
    base.mkdir(parents=True, exist_ok=True)
    return str(base)

def plot_hist(data_np, bins, xlabel, title, out_path):
    plt.figure(figsize=(6.0,4.0))
    plt.hist(data_np, bins=bins, density=True)
    plt.yscale("log")  # tails
    plt.xlabel(xlabel)
    plt.ylabel("density (log scale)")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(f"[saved] {out_path}")

def plot_hexbin(x_np, y_np, gridsize, xlabel, ylabel, title, out_path, max_points_2d=1_000_000, seed=123):
    n = x_np.shape[0]
    if n > max_points_2d:
        rng = np.random.RandomState(seed)
        idx = rng.choice(n, size=max_points_2d, replace=False)
        x_np = x_np[idx]; y_np = y_np[idx]
    plt.figure(figsize=(6.0,5.0))
    hb = plt.hexbin(x_np, y_np, gridsize=gridsize)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    cb = plt.colorbar(hb); cb.set_label("counts")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(f"[saved] {out_path}")


In [9]:

# --- Run: load JSON & checkpoints, then make plots ---
JSON_PATH = "/home/goring/mean_field_langevin/MCMC_sparse/results/plot2/rs_cavity_explicit_aw_ard_P1000_kap7.500e-03_Ptr1000_Peval50000_kap7.500e-03_N512_B1024_g0.5.json"

meta, ckpts, W, a = load_json_and_ckpts(
    JSON_PATH, 
    max_ckpts=MAX_CKPTS, 
    subsample_per_ckpt=SUBSAMPLE_PER_CKPT
)

outdir = ensure_outdir(JSON_PATH, OUTDIR)
print(f"Output dir: {outdir}")

B, d = W.shape
print(f"Aggregated: B={B}, d={d}, total W entries={W.numel()}, total a={a.numel()}")

# 1) p(a)
p_a_path = os.path.join(outdir, "p_a_hist.png")
plot_hist(a.numpy(), BINS, xlabel="a", title="Marginal p(a)", out_path=p_a_path)

# 2) p(w) pooled across all coordinates
p_w_path = os.path.join(outdir, "p_w_hist_pooled.png")
plot_hist(W.reshape(-1).numpy(), BINS, xlabel="w (all coords pooled)", title="Marginal p(w) [pooled]", out_path=p_w_path)

# 3) p(w_j, a) for selected j
js = choose_j_indices(W, j_indices=J_INDICES, topk_by_var=TOPK_BY_VAR)
for j in js:
    x = W[:, j].numpy()
    y = a.numpy()
    p2d_path = os.path.join(outdir, f"p_w{j}_a_hexbin.png")
    plot_hexbin(x, y, gridsize=70, xlabel=f"w[{j}]", ylabel="a",
                title=f"Joint p(w[{j}], a) via hexbin", out_path=p2d_path, max_points_2d=MAX_POINTS_2D)

print("Done.")


loaded    1/9: /home/goring/mean_field_langevin/MCMC_sparse/results/plot2/particles_P1000_kap7.500e-03_iter0000500.pt  ->  W:(1024, 35)  a:(1024,)
loaded    2/9: /home/goring/mean_field_langevin/MCMC_sparse/results/plot2/particles_P1000_kap7.500e-03_iter0001000.pt  ->  W:(1024, 35)  a:(1024,)
loaded    3/9: /home/goring/mean_field_langevin/MCMC_sparse/results/plot2/particles_P1000_kap7.500e-03_iter0001500.pt  ->  W:(1024, 35)  a:(1024,)
loaded    4/9: /home/goring/mean_field_langevin/MCMC_sparse/results/plot2/particles_P1000_kap7.500e-03_iter0002000.pt  ->  W:(1024, 35)  a:(1024,)
loaded    5/9: /home/goring/mean_field_langevin/MCMC_sparse/results/plot2/particles_P1000_kap7.500e-03_iter0002500.pt  ->  W:(1024, 35)  a:(1024,)
loaded    6/9: /home/goring/mean_field_langevin/MCMC_sparse/results/plot2/particles_P1000_kap7.500e-03_iter0003000.pt  ->  W:(1024, 35)  a:(1024,)
loaded    7/9: /home/goring/mean_field_langevin/MCMC_sparse/results/plot2/particles_P1000_kap7.500e-03_iter0003500.pt 